# Figure 2 — 10× dopaminergic interneuron coding

**Populations:** TH and DAT only. **Claims:** odor specificity, signed recruitment, state dependence, and temporal dynamics. Thy1 discrimination is kept in a separate notebook.

## 1. Repository bootstrap and explicit inputs

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from tqdm.auto import tqdm
HERE = Path.cwd().resolve()
REPO = next((p for p in (HERE, *HERE.parents) if (p / 'analysis').is_dir()), None)
if REPO is None: raise RuntimeError('Could not locate the ODyn-analysis repository root.')
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
from analysis.figures.paths import imaging_root, repo_path
MANIFEST = repo_path('analysis', 'stage0', 'ketxyl_16odor_session_manifest.csv')
IMAGING_ROOT = imaging_root()  # set ODYN_IMAGING_ROOT for this computer/server
OUTPUT = repo_path('analysis', 'figures', 'figure2', 'outputs')
OUTPUT.mkdir(parents=True, exist_ok=True)
LINES = {'TH', 'DAT'}
BLANK_ODOR = 0
TAIL_PROBABILITY = 0.01  # descriptive mineral-oil tail, not calibrated trial-level FPR
MIN_GEOMETRY_TRIALS = 3  # per odor, state, and session; >=4 is a sensitivity check
print('Repository:', REPO)
print('Output:', OUTPUT)

## 2. Inventory the actual inputs

The inventory is saved before calculations so missing sessions remain visible.

In [ ]:
from analysis.figures.session_data import available_sessions
inventory = pd.DataFrame(available_sessions(MANIFEST, IMAGING_ROOT, objective='10x'))
inventory['line'] = inventory.population.str.split('-').str[0]
inventory = inventory[inventory.line.isin(LINES)].copy()
inventory.to_csv(OUTPUT / 'da_10x_session_inventory.csv', index=False)
display(inventory[['group_id', 'mouse', 'line', 'available', 'grouped_path']])

## 3. Calculate signed odor-specificity metrics

Trials are reduced with the median. Excitation and suppression breadth are binary calls using separate mineral-oil tails. Lifetime sparseness uses analog signed magnitudes across odors. Threshold-excess E/S balance is not treated as biological magnitude balance.

In [ ]:
from analysis.figures.session_data import load_grouped
from analysis.figures.summaries import signed_session_tables
unit_rows, odor_rows = [], []
signed_inputs = inventory.loc[inventory.available].to_dict('records')
for row in tqdm(signed_inputs, desc='10x signed metrics', unit='session'):
    session = load_grouped(row, row['grouped_path'])
    units, odors = signed_session_tables(session, blank_odor=BLANK_ODOR, tail_probability=TAIL_PROBABILITY, reducer='median')
    unit_rows.extend(units); odor_rows.extend(odors)
unit_metrics = pd.DataFrame(unit_rows); odor_metrics = pd.DataFrame(odor_rows)
unit_metrics.to_csv(OUTPUT / 'da_10x_unit_signed_metrics.csv', index=False)
odor_metrics.to_csv(OUTPUT / 'da_10x_odor_population_metrics.csv', index=False)
print(f'Saved {len(unit_metrics):,} unit-state and {len(odor_metrics):,} odor-state rows.')
display(unit_metrics.head())

## 4. Panel 2A — excitation and suppression breadth

ROIs are first summarized to a session median. Thin lines are sessions; thick estimates average sessions within mouse before taking the across-mouse median.

In [ ]:
import matplotlib.pyplot as plt
session_signed = unit_metrics.groupby(['group_id', 'mouse', 'line', 'state'], as_index=False)[['excitation_breadth', 'suppression_breadth', 'excitation_lifetime_sparseness', 'suppression_lifetime_sparseness']].median()
session_signed.to_csv(OUTPUT / 'da_10x_signed_session_summary.csv', index=False)
colors = {'TH': '#2b8cbe', 'DAT': '#6a51a3'}
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.5), constrained_layout=True)
for ax, metric, title in zip(axes, ['excitation_breadth', 'suppression_breadth'], ['Excitation breadth', 'Suppression breadth']):
    for line in ('TH', 'DAT'):
        selected = session_signed[session_signed.line == line]
        for _, session in selected.groupby('group_id'):
            values = session.set_index('state')[metric]
            if {'pre', 'post'}.issubset(values.index): ax.plot([0, 1], [values.pre, values.post], color=colors[line], alpha=.25)
        mouse = selected.groupby(['mouse', 'state'])[metric].mean().unstack()
        center = mouse[['pre', 'post']].median()
        ax.plot([0, 1], center, color=colors[line], marker='o', lw=2.5, label=line)
    ax.set(xticks=[0, 1], xticklabels=['awake', 'ket/xyl'], ylabel='fraction of odors', title=title)
axes[0].legend(frameon=False)
panel = OUTPUT / 'panel_2A_da_signed_breadth.png'
fig.savefig(panel, dpi=200); plt.show()
print('Saved:', panel)

## 5. Panel 2B — excitatory and suppressive lifetime sparseness

These are lifetime metrics: selectivity across odors within an ROI. They are not population sparsity metrics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.5), constrained_layout=True)
for ax, metric, title in zip(axes, ['excitation_lifetime_sparseness', 'suppression_lifetime_sparseness'], ['Excitatory lifetime sparseness', 'Suppressive lifetime sparseness']):
    for line in ('TH', 'DAT'):
        selected = session_signed[session_signed.line == line]
        for _, session in selected.groupby('group_id'):
            values = session.set_index('state')[metric]
            if {'pre', 'post'}.issubset(values.index): ax.plot([0, 1], [values.pre, values.post], color=colors[line], alpha=.25)
        mouse = selected.groupby(['mouse', 'state'])[metric].mean().unstack()
        center = mouse[['pre', 'post']].median()
        ax.plot([0, 1], center, color=colors[line], marker='o', lw=2.5, label=line)
    ax.set(xticks=[0, 1], xticklabels=['awake', 'ket/xyl'], ylabel='lifetime sparseness', title=title)
axes[0].legend(frameon=False)
panel = OUTPUT / 'panel_2B_da_lifetime_sparseness.png'
fig.savefig(panel, dpi=200); plt.show()
print('Saved:', panel)

## 6. Calculate mixture geometry for TH and DAT only

Each trial is an observation and each glomerular ROI is a feature. Integrated crossnobis retains magnitude/SNR; spatiotemporal crossnobis concatenates four 1-second bins; cosine is a gain-insensitive, non-crossvalidated secondary description.

In [ ]:
from analysis.figures.figure2.make_10x_mixture_geometry import analyze_session
geometry_rows = []
geometry_inputs = inventory.loc[inventory.available].to_dict('records')
for row in tqdm(geometry_inputs, desc='10x DA geometry', unit='session'):
    try: geometry_rows.extend(analyze_session(row, Path(row['grouped_path'])))
    except ValueError as error: print(f"Excluded group {row['group_id']}: {error}")
geometry = pd.DataFrame(geometry_rows)
geometry.to_csv(OUTPUT / 'da_10x_mixture_geometry_long.csv', index=False)
geometry_summary = geometry.drop_duplicates(['group_id', 'mouse', 'line', 'state', 'pair'])
geometry_summary.to_csv(OUTPUT / 'da_10x_mixture_geometry_session_summary.csv', index=False)
geometry_summary['min_trials'] = geometry_summary[['n_a', 'n_b']].min(axis=1)
geometry_excluded = geometry_summary[geometry_summary.min_trials < MIN_GEOMETRY_TRIALS].copy()
geometry_primary = geometry_summary[geometry_summary.min_trials >= MIN_GEOMETRY_TRIALS].copy()
valid_keys = geometry_primary[['group_id', 'state', 'pair']].drop_duplicates()
geometry_primary_long = geometry.merge(valid_keys, on=['group_id', 'state', 'pair'], how='inner')
geometry_excluded.to_csv(OUTPUT / 'da_10x_geometry_low_trial_exclusions.csv', index=False)
print(f'Primary comparisons: {len(geometry_primary)}; low-trial comparisons excluded: {len(geometry_excluded)}')
display(geometry_excluded[['group_id', 'mouse', 'line', 'state', 'pair', 'n_a', 'n_b']])

## 7. Panel 2C — DA mixture geometry

The rows keep magnitude-sensitive, temporal, and gain-insensitive geometry separate.

In [ ]:
from analysis.figures.figure2.make_10x_mixture_geometry import plot_state_comparison
from IPython.display import Image, display
panel = OUTPUT / 'panel_2C_da_mixture_geometry.png'
plot_state_comparison(panel, geometry_primary)
display(Image(filename=str(panel)))
print('Saved:', panel)

## 8. Panel 2D — when DA mixture information emerges

Cumulative crossnobis uses odor onset through 1, 2, 3, or 4 seconds, gaining temporal localization without relying on one noisy short bin.

In [ ]:
from analysis.figures.figure2.make_10x_mixture_geometry import plot_cumulative
panel = OUTPUT / 'panel_2D_da_cumulative_geometry.png'
plot_cumulative(panel, geometry_primary_long)
display(Image(filename=str(panel)))
print('Saved:', panel)